# MCP 05 · 服务端与客户端（FastMCP）

MCP（Model Context Protocol，模型上下文协议）是一种**标准化的工具调用协议**，
可以把它理解成「AI 世界的 USB 接口」：工具作者按 MCP 规范写一次服务端，
Claude Desktop / Cursor / LangChain / DeepAgents / 自研 Agent 都能即插即用。

本节是 MCP 的第一课，把「服务端 → 客户端」这条最短链路跑通，全节四个概念：

| 概念 | 是什么 | 本节的代码形态 |
|---|---|---|
| 工具 Tool | 让模型「做事」的函数 | `@mcp.tool` 装饰器 + 普通函数 |
| 服务端 Server | 把工具注册进去的实例 | `mcp = FastMCP("生活服务")` |
| 客户端 Client | 连上服务端、列工具、调工具 | `async with Client(url) as client` |
| 传输 Transport | 服务端和客户端怎么通信 | `stdio`（子进程）/ `streamable-http`（网络） |

> **本 notebook 由 `Agent/05_mcp/` 下 4 个脚本合并而成**：
> `01_服务端.py`（课案原版 61 行）、`01_服务端_jxsd.py`（完整版 263 行）、
> `02_客户端.py`（课案原版 50 行）、`02_客户端_jxsd.py`（完整版 211 行）。

**官方文档**
- FastMCP 服务端：<https://gofastmcp.com/servers/server>
- FastMCP 客户端：<https://gofastmcp.com/clients/client>
- MCP 规范：<https://modelcontextprotocol.io>

## 运行条件

| 项 | 说明 |
|---|---|
| 🔴 运行档位 | **需外部服务** —— notebook 内自己起一个 MCP 服务端（fastmcp，streamable-http，端口 **8150**），末尾自动关闭 |
| 依赖 | `fastmcp` / `uvicorn`（本项目 venv 已装） |
| 密钥 | 无（本课工具都是本地纯函数，不调大模型） |
| 前置服务 | 无（服务端由 notebook 自己拉起，不必另开窗口） |
| 预计耗时 | 约 15~30 秒 |

> 这是 `05_mcp/` 里**最基础**的一课：它演示「MCP 服务端长什么样、客户端怎么连」。
> 因为要真起一个常驻服务，所以是 🔴 档；但它**不调大模型、不读 `.env`**，
> 工具都是写死的本地函数（查天气、四则运算）。

## 本节地图

先看「服务端 ↔ 客户端」这条链路，以及两种传输方式各走什么路。

```mermaid
graph LR
    A["服务端<br/>FastMCP + @mcp.tool 工具"] -->|"streamable-http<br/>http://127.0.0.1:8150/mcp"| B["客户端<br/>Client(url)"]
    A -->|"stdio<br/>客户端拉起子进程"| C["客户端<br/>StdioTransport"]
    B --> D["list_tools / call_tool"]
    C --> D
```

上面这张图等价于下面这张表（**裸 JupyterLab 不渲染 mermaid，看表即可**）：

| 传输 | 服务端怎么起 | 客户端怎么连 | 适用 |
|---|---|---|---|
| `stdio` | 客户端当**子进程**拉起服务脚本，走标准输入输出 | `StdioTransport(command=..., args=[...])` | 本地桌面客户端（Claude Desktop） |
| `streamable-http` | 作为**网络服务**监听端口 | `Client("http://127.0.0.1:8150/mcp")` | 部署成服务、多客户端共享 |

与上下节的衔接：本课把「服务端 + 客户端」这条最短链路打通；下一课
`02_资源与提示词.ipynb` 会补上 `@mcp.resource`（读数据）和 `@mcp.prompt`（提示词模板）。

## 0. 环境引导

notebook 的**工作目录默认是它自己所在的文件夹**，而本项目代码都写
`from config import settings`（`config.py` 在仓库根）。所以每个 notebook 的第一格
统一做一件事：**向上找到仓库根，切过去，并塞进 `sys.path`**。

> 本课其实用不到 `config`，但这一格仍保留——保持全仓统一，并给出 `NB_DIR` / `WORKDIR`
> 两个变量：本课要把服务端脚本落盘到 `WORKDIR` 下的专属子目录，靠的就是它们。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

## 0.5 前置条件自检 + 落盘服务端脚本

这一格做两件事：
1. **自检**：`fastmcp` / `uvicorn` 装了没、端口 8150 空闲没。缺了就打印中文提示，
   让后面的 cell 走降级路径（不抛异常、不连服务）。
2. **落盘统一服务端脚本**：把「含全部 5 个工具的服务端」写成一个独立的 `server.py`
   存到 `WORKDIR / "mcp_server"`（本课专属子目录），供两种用途：
   - stdio 演示：`StdioTransport` 把它当**子进程**拉起；
   - http 演示：`subprocess.Popen` 把它当**后台进程**拉起。

> 为什么单独写一个 `server.py` 而不是直接在 notebook 里 `mcp.run()`？
> 因为 `mcp.run()` 是常驻服务，直接跑会把内核**永久阻塞**——必须落盘后交给子进程跑。

In [ ]:
# ===== 关键：把 sys.stderr 换成真实文件（必须在 import fastmcp / mcp 之前）=====
# IPython 会把 sys.stderr 换成没有真实 fileno() 的 OutStream；而 mcp 的 stdio_client
# 把 sys.stderr 当默认参数交给子进程用，一用就 UnsupportedOperation: fileno。
# 所以这里先换成真实文件，§2.8 末尾再还回去。
_real_stderr = sys.stderr
sys.stderr = open(str(WORKDIR / "mcp_stderr.log"), "w", encoding="utf-8")

# ---------- 前置条件自检 ----------
import importlib.util
import socket as _socket

NB_HTTP_PORT = 8150   # 本 notebook 独占端口（同章 5 个 notebook 并发，各用各的，别撞）

def _port_in_use(host: str, port: int) -> bool:
    """探测端口是否已被监听：用来判断「外面是不是已经有服务在跑」。"""
    with _socket.socket() as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0

_missing = [m for m in ("fastmcp", "uvicorn") if importlib.util.find_spec(m) is None]
_service_ready = True
if _missing:
    print(f"[跳过] 缺少依赖：{', '.join(_missing)}，请先 `pip install fastmcp uvicorn`")
    _service_ready = False
elif _port_in_use("127.0.0.1", NB_HTTP_PORT):
    print(f"[跳过] 端口 {NB_HTTP_PORT} 已被占用，请先释放（本 notebook 需要 {NB_HTTP_PORT}）")
    _service_ready = False
else:
    print(f"✅ 依赖齐全；端口 {NB_HTTP_PORT} 空闲，可以起服务。")

# ---------- 落盘统一服务端脚本 + notebook 专用辅助函数 ----------
import asyncio
import subprocess
import threading
import time

SERVER_DIR = WORKDIR / "mcp_server"     # 本课专属子目录（同章共享 WORKDIR，必须再套一层）
SERVER_DIR.mkdir(exist_ok=True)
SERVER_SCRIPT_NB = SERVER_DIR / "server.py"

SERVER_PY_SRC = '''# -*- coding: utf-8 -*-
from fastmcp import FastMCP
import sys

mcp = FastMCP("生活服务")


@mcp.tool
def get_weather(city: str) -> str:
    """查询指定城市的实时天气。city：城市名称，如「上海」"""
    weather_map = {"上海": "晴 25 度", "北京": "多云 18 度", "广州": "阵雨 30 度"}
    return weather_map.get(city, f"{city} 天气未知")


@mcp.tool
def add(a: float, b: float) -> float:
    """两数相加"""
    return a + b


@mcp.tool
def sub(a: float, b: float) -> float:
    """两数相减"""
    return a - b


@mcp.tool
def mul(a: float, b: float) -> float:
    """两数相乘"""
    return a * b


@mcp.tool
def div(a: float, b: float) -> float:
    """两数相除"""
    if b == 0:
        raise ValueError("除数不能为 0")
    return a / b


if __name__ == "__main__":
    mode = sys.argv[1].lower() if len(sys.argv) > 1 else ""
    if mode in ("http", "streamable-http", "streamable_http"):
        mcp.run(transport="streamable-http", host="127.0.0.1", port=8150, path="/mcp")
    else:
        mcp.run(transport="stdio")
'''
SERVER_SCRIPT_NB.write_text(SERVER_PY_SRC, encoding="utf-8")
print("服务端脚本已落盘：", SERVER_SCRIPT_NB)

_server_proc = None    # 后台服务进程句柄；后面的启动格填充，最后一个格用 taskkill 关它
_server_ready = False  # 服务是否真的就绪（语义校验通过才算）；demo_http 靠它决定跑不跑


def port_ready(host: str, port: int) -> bool:
    """探测端口是否已监听：轮询等服务就绪用（别写死长 sleep）。"""
    with _socket.socket() as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0


def run_async(coro):
    """在独立线程里 asyncio.run(coro)，异常回抛主线程，正常则返回协程结果。

    ipykernel 主线程已经有一个运行中的事件循环，顶层 asyncio.run() 会抛
    RuntimeError: asyncio.run() cannot be called from a running event loop；
    在子线程里跑就没有这个问题。
    """
    box = {}

    def _target():
        try:
            box["value"] = asyncio.run(coro)
        except BaseException as exc:      # noqa: BLE001
            box["error"] = exc

    t = threading.Thread(target=_target, daemon=True)
    t.start()
    t.join()
    if "error" in box:
        raise box["error"]
    return box.get("value")


async def _mcp_list_tools(url: str):
    """就绪语义校验：真连一次 list_tools，返回工具名清单。

    「端口有人监听」≠「是我们的服务」（可能被别的进程占用）。这里真的发起一次
    list_tools，确认清单里有 get_weather/div 才认定是本课服务端。
    """
    from fastmcp import Client
    async with Client(url) as client:
        tools = await client.list_tools()
        return [t.name for t in tools]

## 1. 课案原版：三行起一个服务端

课案原版只有 61 行，正好够把「服务端 + 客户端」各出现一次。
**先看最短实现，再看完整版**，两者的差距就是本课要讲的全部内容。

### 1.1 服务端：三行核心 + 两个工具

FastMCP 起一个服务端，核心只有三行：

```python
from fastmcp import FastMCP
mcp = FastMCP("生活服务")
@mcp.tool
def add(a: int, b: int) -> int: ...
```

**函数的类型注解 + docstring 就是给模型看的工具说明书**：FastMCP 会把签名转成
JSON Schema（参数名/类型/必填），把 docstring 转成 description。注解写错 → 模型调用就错。

In [ ]:
# ---------- 1.1 服务端原版：三行核心 + 两个工具 ----------
from fastmcp import FastMCP

# 创建 MCP 服务实例。name 会显示在客户端的工具列表里
mcp = FastMCP(name="生活服务")


# ---------- 工具 ----------
@mcp.tool
def get_weather(city: str) -> str:
    """查询指定城市的实时天气。city：城市名称，如「上海」"""
    weather_map = {"上海": "晴 25 度", "北京": "多云 18 度", "广州": "阵雨 30 度"}
    return weather_map.get(city, f"{city} 天气未知")


@mcp.tool
def add(a: int, b: int) -> int:
    """计算两个整数的和"""
    return a + b


def run_stdio():
    """stdio 模式：被客户端以子进程方式拉起（如 02_langchain/12 的用法）"""
    mcp.run()  # 默认 stdio


def run_http():
    """HTTP 模式：作为独立网络服务运行，客户端通过 URL 连接"""
    mcp.run(transport="http", host="127.0.0.1", port=8000, path="/mcp")


def _cli_main():
    if len(sys.argv) > 1 and sys.argv[1] == "http":
        run_http()
    else:
        run_stdio()

说明两点：
- `run_stdio()` / `run_http()` 是两种**传输方式**的启动入口（原版只写了这两种）；
  完整版 §2 会补全成四种。
- `_cli_main()` 是原版命令行入口的等价物：按第一个参数选传输方式。notebook 里**不调用它**
  （`mcp.run(...)` 会阻塞内核），保留这段只为让你看到「同一个文件既能当示例又能当服务端」的原样写法。

### 1.2 客户端：stdio + http 两种连接

服务端起好了，客户端只需要「一个 transport 或一个 URL」就能连上去。原版客户端只做两件事：
`list_tools()`（列工具）和 `call_tool()`（调工具）。

In [ ]:
# ---------- 1.2 客户端原版：stdio + http 两种连接 ----------
import asyncio
from fastmcp import Client
from fastmcp.client.transports import StdioTransport, StreamableHttpTransport


async def demo_stdio():
    """本地 stdio：自动拉起 01_服务端.py 子进程"""
    transport = StdioTransport(
        command="uv", args=["run", str(Path(__file__).resolve().parent / "01_服务端.py")],
    )
    async with Client(transport) as client:
        tools = await client.list_tools()
        print("可用工具：", [t.name for t in tools])
        result = await client.call_tool("get_weather", {"city": "上海"})
        print("调用结果：", result)


async def demo_http():
    """远程 HTTP：先启动服务（uv run 05_mcp/01_服务端.py http）"""
    transport = StreamableHttpTransport(url="http://127.0.0.1:8000/mcp")
    async with Client(transport) as client:
        result = await client.call_tool("add", {"a": 1, "b": 2})
        print("HTTP 调用结果：", result)


def _cli_main():
    asyncio.run(demo_stdio())
    # 需要 HTTP 服务已启动，否则注释掉下一行
    # asyncio.run(demo_http())

这里有两个**原版写法在 notebook 里不能直接用**的点（完整版 §2 都会改掉）：

1. `demo_stdio` 用 `command="uv", args=["run", str(Path(__file__)...)]` 定位服务脚本——
   notebook 里没有 `__file__`，`uv` 也未必在 PATH 上；
2. `demo_http` 把 URL 写死成 `127.0.0.1:8000`——本课的服务端要用 8150（8000 会被同章别的课占用）。

### 1.3 跑一下原版的 stdio 连接（notebook 适配版）

先跑 **stdio** 这条（它不需要预先起服务，客户端会自己把服务脚本当子进程拉起来）。
下面把原版那句 `uv run` + `Path(__file__)` 改成 `sys.executable` + 已落盘的 `server.py`
（§0.5 已写好），其余逻辑不变。

In [ ]:
# ---------- 1.3 跑一下原版的 stdio 连接（notebook 适配版） ----------
async def _demo_stdio_nb() -> None:
    transport = StdioTransport(command=sys.executable, args=[str(SERVER_SCRIPT_NB)])
    async with Client(transport) as client:
        tools = await client.list_tools()
        print("可用工具：", [t.name for t in tools])
        result = await client.call_tool("get_weather", {"city": "上海"})
        print("调用结果：", result.content[0].text)


if _service_ready:
    run_async(_demo_stdio_nb())
else:
    print("[跳过] 前置条件未就绪，跳过 stdio 演示。")

### 预期输出

```text
可用工具： ['get_weather', 'add', 'sub', 'mul', 'div']
调用结果： 晴 25 度
```

三个值得停一下的点：
- 工具列表里有 **5 个**（`get_weather` + 四则运算）——因为落盘的 `server.py` 是「原版 + 完整版」的并集；
- `get_weather("上海")` 返回 `晴 25 度`，正是服务端 `weather_map` 里的值；
- stdio 模式下服务端是**子进程**，跑完这段它就被 `async with` 关掉了，没有残留。

## 2. 完整版：四个工具 + 四种传输

课案原版只有「查天气 + 加法」两个工具、`stdio`/`http` 两种传输。
完整版把工具补成四则运算（`add/sub/mul/div`），把传输补成四种，并讲清各自的取舍。

### 2.1 服务端：四个工具，含一个会抛异常的 `div`

注意 `div` 里唯一的业务异常：除零时 `raise ValueError("除数不能为 0")`。
MCP 会把这个异常包成 `is_error=True` 的结果回给客户端（而不是让连接崩掉），
模型看到错误信息后可以自己纠正参数重试——这是「工具报错」的正确姿势。

In [ ]:
# ---------- 2.1 服务端完整版：四个工具 ----------
from fastmcp import FastMCP

# name 会出现在客户端的服务列表里（课案里叫 "演示 🚀"，emoji 是故意加的：
# 用来验证从服务端 → 传输 → 客户端整条链路的 UTF-8 编码是否正常）。
mcp = FastMCP("演示 🚀")


@mcp.tool
def add(a: float, b: float) -> float:
    """两数相加
    :param a:第一个数字
    :param b:第二个数字
    :return: a+b的结果
    """
    return a + b


@mcp.tool
def sub(a: float, b: float) -> float:
    """两数相减
    :param a:第一个数字
    :param b:第二个数字
    :return: a-b的结果
    """
    return a - b


@mcp.tool
def mul(a: float, b: float) -> float:
    """两数相乘
    :param a:第一个数字
    :param b:第二个数字
    :return: a*b的结果
    """
    return a * b


@mcp.tool
def div(a: float, b: float) -> float:
    """两数相除
    :param a:第一个数字
    :param b:第二个数字
    :return: a/b的结果
    """
    # 除零是「工具里唯一的业务异常」。MCP 会把抛出的异常包成 is_error=True 的结果
    # 回给客户端（而不是让整条连接崩掉），模型看到错误信息后可以自己纠正参数重试。
    if b == 0:
        raise ValueError("除数不能为 0")
    return a / b


# 本文件对外服务的地址（与 02_客户端_jxsd.py 约定一致）
HTTP_HOST = "127.0.0.1"
HTTP_PORT = 8000
MCP_PATH = "/mcp"

### 2.2 四种传输方式怎么启动

同一份 `mcp` 实例，换一个 `transport` 参数就是另一种启动方式：

| 传输 | 启动写法 | 特点 |
|---|---|---|
| `stdio` | `mcp.run(transport="stdio")` | 本地子进程，客户端拉起本脚本，走标准输入输出 |
| `http` | `mcp.run(transport="http", port=8000, host="0.0.0.0")` | 普通请求/响应，类 REST，不支持流式 |
| `streamable-http` | `mcp.run(transport="streamable-http", ...)` | 全双工，可流式推送中间状态（**生产推荐**） |
| `sse` | `mcp.run(transport="sse", port=8001, host="0.0.0.0")` | 半双工，**已弃用**，被 streamable-http 取代 |

选哪个？
- 本机给 Claude Desktop / LangChain 用 → `stdio`（省去端口和鉴权，客户端帮你拉进程）；
- 部署成网络服务 / 多客户端共享 → `streamable-http`（可流式、可挂中间件、可多进程）；
- 老客户端兼容 → 才考虑 `sse`，新项目别用。

In [ ]:
# ---------- 2.2 四种传输的启动函数 ----------
def run_stdio() -> None:
    """方式一：stdio —— 不监听端口，靠标准输入输出和父进程说话。"""
    mcp.run(transport="stdio")


def run_http() -> None:
    """方式二：http —— 普通请求/响应，类 REST，不支持流式。"""
    mcp.run(transport="http", host=HTTP_HOST, port=HTTP_PORT, path=MCP_PATH)


def run_streamable_http() -> None:
    """方式三：streamable-http —— 全双工、可流式（课案默认推荐，也是本套文件默认）。"""
    mcp.run(transport="streamable-http", host=HTTP_HOST, port=HTTP_PORT, path=MCP_PATH)


def run_sse() -> None:
    """方式四：sse —— 半双工，已弃用，只作教学对照。"""
    mcp.run(transport="sse", host=HTTP_HOST, port=8001)

完整版还带了一个 `self_check()`：**无参数运行时**它在后台线程里起一个 streamable-http 服务、
自己连自己看一眼（`list_tools` + 调一次 `div`），跑完自动收摊。这样学员「只运行一个文件」
就能确认服务端真的起来了。下面是它的原样代码（notebook 里**不调用**，改用下面的 subprocess 方式起服务）：

In [ ]:
# ---------- 2.2（附）单文件自检 self_check：线程式起服务 + 自连自探 ----------
def self_check() -> None:
    """无参数运行时的自检演示：后台线程起 streamable-http，自己连自己看一眼。"""
    import threading
    import time

    import uvicorn
    from fastmcp import Client

    # mcp.http_app() 把 FastMCP 实例包成一个标准 ASGI 应用，
    # 于是可以交给任意 ASGI 服务器（uvicorn / hypercorn）跑，也能挂 --workers 多进程。
    app = mcp.http_app(transport="streamable-http", path=MCP_PATH)

    server = uvicorn.Server(
        uvicorn.Config(app, host=HTTP_HOST, port=HTTP_PORT, log_level="warning")
    )
    thread = threading.Thread(target=server.run, daemon=True)   # daemon：主线程退出时自动收摊
    thread.start()

    # 等服务真正就绪（最多 10 秒）。直接 sleep 固定秒数是不可靠的写法。
    for _ in range(100):
        if server.started:
            break
        time.sleep(0.1)

    # 起不来最常见的原因就是端口被占用（比如上一个窗口的服务端没关干净）。
    # 这里**不能**直接抛异常：自检失败也要给出人能看懂的中文提示。
    if not server.started:
        print(f"❌ 服务未能启动：{HTTP_HOST}:{HTTP_PORT} 可能已被占用。")
        print("   请换端口，或先关掉占用该端口的程序。")
        return

    print(f"✅ 服务端已启动：http://{HTTP_HOST}:{HTTP_PORT}{MCP_PATH}（传输：streamable-http）")

    # 局部 import asyncio：只有自检这条路才需要事件循环，常驻服务那几条路用不上
    import asyncio

    async def _probe() -> None:
        # 课案原文写法就是直接给 URL：Client("http://localhost:8000/mcp")
        # FastMCP 会按 URL 自动选出 streamable-http 传输。
        async with Client(f"http://{HTTP_HOST}:{HTTP_PORT}{MCP_PATH}") as client:
            # 自检的核心就是这两件事：工具注册上了没有（list_tools）、
            # 工具能跑通没有（call_tool）。两件都过，说明服务端真的可用。
            tools = await client.list_tools()
            print(f"   已注册工具 {len(tools)} 个：")
            for t in tools:
                # inputSchema 就是 FastMCP 从类型注解自动生成的 JSON Schema，
                # 模型靠它知道该传什么参数。description 来自函数 docstring 首行。
                params = list((t.inputSchema or {}).get("properties", {}).keys())
                print(f"     - {t.name}({', '.join(params)})：{t.description}")
            # 挑 div 而不是 add：因为 div 有除零分支，能顺带验证「工具返回值取法」这条路
            result = await client.call_tool("div", {"a": 10, "b": 4})
            print(f"   调用 div(10, 4) = {result.content[0].text}")

    asyncio.run(_probe())

    # 收摊：先让 uvicorn 停，再等线程结束。顺序反了会打印到一半日志就断掉
    server.should_exit = True       # 通知 uvicorn 退出
    thread.join(timeout=10)         # 等它真的退完；daemon 线程不等也不会阻塞主线程，但日志会乱序
    print("✅ 服务端已关闭，演示结束。")

    # 自检跑完顺带把「怎么当常驻服务用」再讲一遍：
    # 学员看完输出就知道下一步该敲哪条命令，不用回去翻文件头
    print()
    print("要把本文件当常驻服务用，请选择传输方式：")
    print(f"    uv run Agent/05_mcp/01_服务端_jxsd.py http   → http://{HTTP_HOST}:{HTTP_PORT}{MCP_PATH}")
    print("    uv run Agent/05_mcp/01_服务端_jxsd.py sse    → http://127.0.0.1:8001/sse（已弃用）")
    print("    uv run Agent/05_mcp/01_服务端_jxsd.py stdio  → 交给客户端当子进程拉起")


def _cli_main():
    mode = sys.argv[1].lower() if len(sys.argv) > 1 else ""
    if mode == "http":
        run_http()
    elif mode in ("streamable-http", "streamable_http"):
        run_streamable_http()
    elif mode == "sse":
        run_sse()
    elif mode == "stdio":
        run_stdio()
    else:
        self_check()

### 2.3 客户端：连 HTTP 服务、拆 CallToolResult、故意触发除零

完整版客户端讲清三个最容易踩的点：

1. **一切都是异步的**：`Client` 必须 `async with` 进入上下文，且 FastMCP 3.x 起
   不支持同步用法，入口永远是 `asyncio.run(main())`。
2. **客户端只认「工具名 + 参数字典」**：`call_tool("add", {"a": 1, "b": 2})`，
   参数名必须和服务端函数签名一致，写错会得到 `is_error=True` 的结果。
3. **返回的不是裸值，是 `CallToolResult`**：
   - `result.content[0].text` → 文本（MCP 协议层给的就是文本）
   - `result.data` → FastMCP 帮你按返回类型反序列化后的 Python 值
   - `result.is_error` → 这次调用是否出错

In [ ]:
# ---------- 2.3 客户端完整版：自检脚手架 + http/stdio 两个演示 ----------
import asyncio
from pathlib import Path
import socket
import sys

import threading
import time

import uvicorn
from fastmcp import Client

# 与服务端约定的地址，和 01_服务端_jxsd.py 保持一致
try:
    SERVER_SCRIPT = Path(__file__).resolve().parent / "01_服务端_jxsd.py"
except NameError:
    SERVER_SCRIPT = None   # notebook 里没有 __file__；§2.4 会重指到已落盘的 server.py
HTTP_HOST = "127.0.0.1"
HTTP_PORT = 8000
MCP_URL = f"http://{HTTP_HOST}:{HTTP_PORT}/mcp"


def port_in_use(host: str, port: int) -> bool:
    """探测端口是否已被监听：用来判断「外面是不是已经有一个服务端在跑」。"""
    sock = socket.socket()
    sock.settimeout(0.5)         # 短超时：探测不该让脚本卡住
    try:
        sock.connect((host, port))   # 能连上就说明有人监听
        return True
    except OSError:
        return False                 # 连不上不是错误，只是「没人监听」
    finally:
        sock.close()


def start_server_in_thread():
    """把 01 的服务端在本进程的后台线程里拉起来，返回 (uvicorn.Server, Thread) 或 None。"""
    if port_in_use(HTTP_HOST, HTTP_PORT):
        print(f"ℹ️  检测到 {HTTP_HOST}:{HTTP_PORT} 已有服务在运行，直接连它（不再另起服务端）。")
        return None              # None 的语义是「不是我起的」，退出时不能去关它

    # 动态加载 01 的模块：文件名以数字开头，没法 `import 01_服务端_jxsd`，
    # 用 importlib 从文件路径加载是最干净的做法（不会触发它 __main__ 里的自检）。
    import importlib.util

    spec = importlib.util.spec_from_file_location("mcp_server_01", SERVER_SCRIPT)
    module = importlib.util.module_from_spec(spec)
    sys.modules["mcp_server_01"] = module     # 注册进 sys.modules，dataclass/pydantic 才认得出它
    spec.loader.exec_module(module)

    # 同一个 FastMCP 实例既能 mcp.run()，也能交出 ASGI 应用 —— 后者才能塞进线程里跑
    app = module.mcp.http_app(transport="streamable-http", path="/mcp")
    server = uvicorn.Server(
        uvicorn.Config(app, host=HTTP_HOST, port=HTTP_PORT, log_level="warning")
    )
    thread = threading.Thread(target=server.run, daemon=True)   # daemon：主线程退出即收摊
    thread.start()

    for _ in range(100):        # 最多等 10 秒
        # 轮询等就绪：uvicorn 把「服务真的开始监听」暴露成 server.started，
        # 比写死 time.sleep(2) 可靠 —— 机器慢时 2 秒不够，快时白等。
        if server.started:
            return server, thread
        time.sleep(0.1)

    print(f"❌ 服务端启动失败：{HTTP_HOST}:{HTTP_PORT} 无法监听。")
    return None


async def demo_http() -> None:
    """① 课案原文写法：直接给 URL，FastMCP 按 URL 自动选传输。"""
    print("=" * 64)
    print("① HTTP 连接（课案原文写法：Client(\"http://localhost:8000/mcp\")）")
    print("=" * 64)

    # async with 进入时会：建连 → initialize 握手 → 协商协议版本与能力；
    # 退出时自动关闭连接。不写 async with 手动 new 出来的 client 是不完整的。
    async with Client(MCP_URL) as client:
        # --- list_tools：拿到服务端所有工具的「定义」（不含执行） ---
        tools = await client.list_tools()
        print(f"可用工具 {len(tools)} 个：")
        for tool in tools:
            params = ", ".join((tool.inputSchema or {}).get("properties", {}).keys())
            print(f"  - {tool.name}({params})：{tool.description}")
            # 注释掉课案原始的 print(tool)：它会打印整个 Tool 对象（含完整 JSON Schema），
            # 信息量太大，上面这行挑重点更好读。想看全貌就把下面这行的注释放开。
            # print(tool)

        # --- call_tool：真正执行一个工具 ---
        result = await client.call_tool("add", {"a": 1, "b": 2})
        print(f"\nadd(1, 2) = {result.content[0].text}")
        print(f"  · result.data（反序列化后的 Python 值）= {result.data!r}")
        print(f"  · result.is_error = {result.is_error}")

        # 服务端抛异常时不会断连，而是回一个 is_error=True 的结果。
        # raise_on_error=False 让客户端把错误当「普通返回值」拿回来，而不是抛 Python 异常。
        print("\n下面这条会故意触发服务端异常（服务端控制台会打红色日志，属正常）：")
        bad = await client.call_tool("div", {"a": 1, "b": 0}, raise_on_error=False)
        print(f"div(1, 0) 的返回：is_error={bad.is_error}，内容={bad.content[0].text}")


async def demo_stdio() -> None:
    """② stdio 连接：客户端用 StdioTransport 把服务端脚本当子进程拉起来。"""
    print()
    print("=" * 64)
    print("② stdio 连接（服务端当子进程，无需端口）")
    print("=" * 64)

    from fastmcp.client.transports import StdioTransport

    # 用当前解释器直接跑脚本，避免依赖 uv 在 PATH 里。
    # ⚠️ 第二个参数 "stdio" 必须带上：不传参数时 01 会走「自检演示」分支去抢 8000 端口，
    #    而 stdio 模式下服务端**绝对不能往 stdout 打印任何东西**——
    #    stdout 是 JSON-RPC 的通道，多打印一行字就会把协议流打断。
    transport = StdioTransport(
        command=sys.executable, args=[str(SERVER_SCRIPT), "stdio"]
    )
    # 把 transport 交给 Client 而不是 URL —— 这是 stdio 与 HTTP 在客户端侧唯一的写法差异
    async with Client(transport) as client:
        tools = await client.list_tools()
        print(f"通过 stdio 拿到工具：{[t.name for t in tools]}")
        result = await client.call_tool("mul", {"a": 8, "b": 2})
        print(f"mul(8, 2) = {result.content[0].text}")


async def main() -> None:
    # 顺序有讲究：先跑 HTTP（顺带验证自己起的服务端可用），再跑 stdio（独立子进程，不受影响）
    await demo_http()
    await demo_stdio()


def _cli_main():
    # started 为 None 有两种可能：① 端口已被占用（用了别人的服务端）；② 启动失败。
    # 两种情况下都不该由本文件去关它，所以后面只判 `if started`。
    started = start_server_in_thread()

    try:
        asyncio.run(main())
    finally:
        # 只有「我们自己起的」服务端才需要关；外面已经在跑的那个不能动。
        if started:
            server, thread = started
            server.should_exit = True     # 通知 uvicorn 优雅退出
            thread.join(timeout=10)       # 等线程真正结束，否则最后一行提示可能先于日志打印
            print("\n✅ 本文件启动的服务端已关闭。")

### 2.4 notebook 适配：端口 8000 → 8150、脚本路径 → 已落盘的 `server.py`

完整版客户端代码里有两处**硬编码**依赖原版运行方式，notebook 里要重指一下：
- `SERVER_SCRIPT = Path(__file__)...` → 没有 `__file__`，改成已落盘的 `server.py`；
- `HTTP_PORT = 8000` → 本课独占 **8150**（8000 会被同章别的课占用）。

只改这三行，后面 `demo_http` / `demo_stdio` 里的 `MCP_URL` / `SERVER_SCRIPT` 是**运行时**才读的全局量，
改完即刻生效，函数体一个字都不用动。

In [ ]:
# ---------- 2.4 notebook 适配：端口 + 脚本路径 ----------
HTTP_PORT = NB_HTTP_PORT                          # 8000 → 8150（同章并发，本课独占）
SERVER_SCRIPT = SERVER_SCRIPT_NB                  # Path(__file__) → 已落盘的 server.py
MCP_URL = f"http://{HTTP_HOST}:{HTTP_PORT}/mcp"   # 重新按新端口算 URL

print("MCP_URL =", MCP_URL)
print("SERVER_SCRIPT =", SERVER_SCRIPT)

### 2.5 后台起服务：`subprocess.Popen` 拉起 `server.py`，轮询等就绪 + 语义校验

完整版自带的 `start_server_in_thread()` 是在**线程**里起 uvicorn；notebook 里改用
**子进程**（`subprocess.Popen`）起，好处是：服务进程完全独立于内核，最后能用
`taskkill` 干净地连进程树一起收掉（线程版没有进程边界，收不干净）。

⚠️ 就绪判断不能只靠「端口有人监听」——端口上可能是别人的服务。所以等端口打开后，
再真的连一次 `list_tools` 做**语义校验**：清单里有 `get_weather` / `div` 才算我们的服务。

In [ ]:
# ---------- 2.5 后台起服务（subprocess.Popen + 轮询等就绪 + 语义校验） ----------
_server_log = None
_server_ready = False
if _service_ready:
    _server_env = {**os.environ, "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8",
                   "NO_PROXY": "127.0.0.1,localhost"}
    _server_log = open(SERVER_DIR / "server.log", "w", encoding="utf-8")
    _server_proc = subprocess.Popen(
        [sys.executable, str(SERVER_SCRIPT_NB), "http"],
        cwd=str(ROOT), env=_server_env,
        stdout=_server_log, stderr=subprocess.STDOUT,
    )

    # 等端口就绪：进程必须还活着（死了说明起失败；端口被别人的服务占了也不能算就绪）
    for _ in range(60):                      # 最多等 30 秒
        if _server_proc.poll() is not None:
            break
        if port_ready("127.0.0.1", NB_HTTP_PORT):
            _server_ready = True
            break
        time.sleep(0.5)

    # 语义校验：光「端口有人监听」不够——监听的可能不是我们的服务。
    # 真的连一次 list_tools，确认清单里有 get_weather/div 才是本课服务端。
    if _server_ready:
        try:
            _probe_names = run_async(_mcp_list_tools(f"http://127.0.0.1:{NB_HTTP_PORT}/mcp"))
            if "get_weather" in _probe_names and "div" in _probe_names:
                print(f"✅ 服务端已启动：http://127.0.0.1:{NB_HTTP_PORT}/mcp"
                      f"（streamable-http，已确认 {len(_probe_names)} 个工具）")
                print("   服务端日志：", SERVER_DIR / "server.log")
            else:
                _server_ready = False
                print(f"❌ 端口 {NB_HTTP_PORT} 上监听的不是本课服务端（工具清单：{_probe_names}）")
        except Exception as exc:             # noqa: BLE001
            _server_ready = False
            print(f"❌ 就绪语义校验失败：{type(exc).__name__}: {exc}")

    if not _server_ready:
        print(f"❌ 服务端启动失败（日志见 {SERVER_DIR / 'server.log'}）")
else:
    print("[跳过] 前置条件未就绪，跳过起服务。")

### 预期输出

```text
✅ 服务端已启动：http://127.0.0.1:8150/mcp（streamable-http，已确认 5 个工具）
```

服务端日志写到 `WORKDIR/mcp_server/server.log`（`WORKDIR` 是 notebook 所在目录的 `tmp_nb_work` 子目录，随机器路径变化，故不贴绝对路径）。

### 2.6 客户端连 HTTP：列工具 + `add` + 故意触发除零

服务已经起来了，现在跑完整版客户端的 **HTTP** 演示：先 `list_tools` 看服务端注册了哪些工具，
再调 `add(1, 2)` 看返回值怎么拆，最后故意 `div(1, 0)` 看服务端异常怎么变成 `is_error=True` 的结果。

In [ ]:
# ---------- 2.6 客户端连 HTTP（list_tools + add + 故意除零） ----------
if _service_ready and _server_ready:
    run_async(demo_http())
else:
    print("[跳过] 服务端未就绪，跳过 HTTP 演示。")

### 预期输出

```text
① HTTP 连接（课案原文写法：Client("http://localhost:8000/mcp")）
================================================================
可用工具 5 个：
  - get_weather(city)：查询指定城市的实时天气。city：城市名称，如「上海」
  - add(a, b)：两数相加
  - sub(a, b)：两数相减
  - mul(a, b)：两数相乘
  - div(a, b)：两数相除

add(1, 2) = 3.0
  · result.data（反序列化后的 Python 值）= 3.0
  · result.is_error = False

下面这条会故意触发服务端异常（服务端控制台会打红色日志，属正常）：
div(1, 0) 的返回：is_error=True，内容=Error calling tool 'div': 除数不能为 0
```

### 2.7 客户端连 stdio：不占端口，把服务脚本当子进程

再跑 **stdio** 演示。和 HTTP 的区别：不需要端口、不需要鉴权、进程生命周期跟着客户端走
（Claude Desktop 这类桌面客户端用的就是这个模式）。这次调 `mul(8, 2)`。

In [ ]:
# ---------- 2.7 客户端连 stdio（mul，服务脚本当子进程） ----------
if _service_ready:
    run_async(demo_stdio())
else:
    print("[跳过] 前置条件未就绪，跳过 stdio 演示。")

### 预期输出

```text
② stdio 连接（服务端当子进程，无需端口）
================================================================
通过 stdio 拿到工具：['get_weather', 'add', 'sub', 'mul', 'div']
mul(8, 2) = 16.0
```

### 2.8 收尾：关掉后台起的服务端

Windows 上 `taskkill /F /T /PID` 会**连同子进程树一起**强杀，只杀父进程会留孤儿。
杀掉后文件句柄不会立刻释放，所以这里只关进程、**不去删** `mcp_server/` 临时目录
（删的话会 `WinError 32`，要重试 + 容忍失败，教学脚本里没必要）。

In [ ]:
# ---------- 2.8 收尾：关掉后台起的服务端（连子进程树一起收） ----------
if _server_proc is not None and _server_proc.poll() is None:
    subprocess.run(["taskkill", "/F", "/T", "/PID", str(_server_proc.pid)],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
    time.sleep(1)
    print("✅ 服务端已关闭。")
    if _server_log is not None:
        _server_log.close()
else:
    print("（服务端本就未启动或已退出，无需关闭）")

# 还回 IPython 的 stderr（§0.5 为避开 stdio 传输的 fileno 坑换成了真实文件）
sys.stderr = _real_stderr

## 小结

- **服务端**三行核心：`FastMCP(...)` 建实例 → `@mcp.tool` 把函数变工具 → 选一种 transport 跑起来；
- **类型注解 + docstring 就是工具说明书**：FastMCP 自动生成 JSON Schema 和 description，写错模型就调错；
- **两种传输**：`stdio`（客户端当子进程拉起，本地桌面用）/ `streamable-http`（网络服务，生产推荐）；
- **客户端只认「工具名 + 参数字典」**：`call_tool("add", {"a": 1, "b": 2})`，参数名要和签名一致；
- **返回的是 `CallToolResult`**：`result.content[0].text` 是文本、`result.data` 是反序列化值、
  `result.is_error` 标记是否出错；
- **工具抛异常不会断连**：服务端 `raise` → 客户端拿到 `is_error=True` 的结果，模型可自己纠正重试。

下一课 `02_资源与提示词.ipynb` 会看到 `@mcp.resource`（让模型「读数据」）和
`@mcp.prompt`（可复用提示词模板）——和 `@mcp.tool` 并列的另外两种装饰器。

## 常见坑

1. **`mcp.run()` 会永久阻塞内核**：它是常驻服务。notebook 里要么落盘后 `subprocess.Popen` 起子进程，
   要么像 `self_check()` 那样起后台线程，绝不能顶格直接调。
2. **`asyncio.run()` 在 notebook 里会抛 `RuntimeError`**：ipykernel 主线程已经有运行中的事件循环，
   必须在独立子线程里 `asyncio.run()`（见 `run_async` 辅助函数）。
3. **stdio 模式服务端不能往 stdout 打印任何东西**：stdout 是 JSON-RPC 通道，多打印一行字就把协议流打断。
   所以 `StdioTransport` 的 args 里必须带 `"stdio"`，别让服务脚本走「自检演示」分支去抢端口。
4. **端口别用 8000/9000**：同章 5 个 notebook 并发，各用各的高位端口（本课 8150），撞了会连到别人的服务。
5. **`Path(__file__)` 在 notebook 里不存在**：一律改成 `WORKDIR` 下已落盘的脚本路径。
6. **服务端异常是「正常返回值」**：`div(1, 0)` 不抛客户端异常，而是 `is_error=True` 的结果；
   `raise_on_error=False` 让客户端把错误当普通值拿回来。

## 官方链接

- FastMCP 服务端（Servers）：<https://gofastmcp.com/servers/server>
- FastMCP 客户端（Clients）：<https://gofastmcp.com/clients/client>
- FastMCP 工具（Tools）：<https://gofastmcp.com/servers/tools>
- FastMCP 传输（Transports）：<https://gofastmcp.com/servers/transports>
- MCP 协议规范：<https://modelcontextprotocol.io>